# PARTE 2: API de lluvias 

**Enunciado**: Van a usar la API gratuita de Open-Meteo, que no necesita clave. (Revisar Sesión 5)

## 2.1. Paso 1

**Enunciado**: Con pd.read_html, lean la tabla de departamentos de Wikipedia y quédense con las columnas departamento y capital.

*Pista: Wikipedia rechaza pd.read_html(url) directo (error 403). Primero descarguen la página con requests.get(url, headers={"User-Agent": "Mozilla/5.0"}) y luego pásenle el texto a pd.read_html(io.StringIO(respuesta.text)).*

In [1]:
# Primero, se deben abir los paquetes necesarios para la ejecución del código. 

import pandas as pd
import requests
from io import StringIO

In [2]:
# Primero, se debe definir el enlace.

url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

# Ya en el enunciado nos indican que Wikipedia rechaza el pd.read(url). Por lo cual, a sugerencia de claude, se aplicará el User-Agent.

headers = {"User-Agent": "Mozilla/5.0"}
respuesta = requests.get(url, headers=headers, timeout=30)
respuesta.status_code

200

In [3]:
# para las tablas de la página de Wikipedia, se puede usar pd.read_html, el cual nos brinda la lista de data framen.

tablas = pd.read_html(StringIO(respuesta.text))
print(len(tablas))

tablas[1].head() # la segunda tabla (1), es donde se encuentran los departamentos y capitales.

6


,Bandera,Escudo,Departamento,Nombre oficial,Abreviatura Nacional,Capital,Ciudad más poblada,Superficie (km²),Población (2025),Creación,Ubicación
0,NaN,NaN,Amazonas,Departamento de Amazonas,AMA,Chachapoyas,Bagua Grande,39 200,467 688,21 de noviembre de 1832 (193 años),NaN
1,NaN,NaN,Áncash,Departamento de Áncash,ANC,Huaraz,Chimbote,35 900,1 205 923,12 de febrero de 1821 (205 años)[nota 1],NaN
2,NaN,NaN,Apurímac,Departamento de Apurímac Apurimaq suyu,APU,Abancay,Andahuaylas,20 900,467 513,28 de abril de 1873 (153 años),NaN
3,NaN,NaN,Arequipa,Departamento de Arequipa Ariqipa suyu,ARE,Arequipa,Arequipa,63 300,1 814 554,26 de abril de 1822 (204 años),NaN
4,NaN,NaN,Ayacucho,Departamento de Ayacucho,AYA,Ayacucho,Ayacucho,43 800,709 026,26 de abril de 1822 (204 años)[nota 2],NaN


In [4]:
# Ahora, de acuerdo con el anuncio, se debe extraer información de las columnas departamento y capital, y cambiaro los nombes a "departamento" y "capital". Para ello, se usará copy(), para crear una nueva tabla.

capitales = tablas[1][["Departamento", "Capital"]].copy()
capitales.columns = ["departamento", "capital"]   # en minúsculas, como pide el enunciado
print(capitales.shape)
capitales


(24, 2)


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## 2.2. Paso 2

Verificación: la tabla final debe tener 25 filas (los 24 departamentos y el Callao). Si falta alguno,agréguenlo y expliquen de dónde lo sacaron.

*Ojo: revisen también la columna capital. Algunas celdas traen texto extra (por ejemplo, lacapital de Lima aparece como "Huacho (de facto)") y así la API no la va a encontrar. Decidan qué ciudad usar y explíquenlo.*

De la tabla creada, se identifica que se registraron 24 departamentos, no 25, ya que no contabilizó a Callao. Además, la tabla indica que la capital de Lima es Huacho. Esto no es error del comando, sino que así está registrado en la página de Wikipedia. 

Primero, se va a agregar la información de Callao a la tabla capitales. Este dato se tomará de otra tabla del mismo artículo de Wikipedia (la tabla de provincias con régimen especial: Callao y Lima).

In [5]:
# Primero se debe determinar de donde se sacará la información de Callao, por lo cual, se debe buscar a a través de la variable tabla, definida en el primer paso.

tablas[3][["Provincia", "Capital"]] #Se intentó con la tabla 2, pero no se encontró la información de Callao, por lo cual, se intentó con la tabla 3.

,Provincia,Capital
0,Callao[nota 6],Callao
1,Lima[nota 7],Lima


In [6]:
# Para agregar la información de Callao, primero se debe crear su data frame de la tabla 3. 

callao = tablas[3][["Provincia", "Capital"]].copy()
callao.columns = ["departamento", "capital"]

# Por recomendación de Claude, se retiró la nota que estaba vinculada a la palabra Callao.

callao["departamento"] = callao["departamento"].str.replace(r"\[.*\]", "", regex=True)

# Luego se selecciona solo con el Callao

callao = callao[callao["departamento"] == "Callao"]
callao

,departamento,capital
0,Callao,Callao


In [7]:
# Luego, se debe agregar la información del Callao a la tabla capitales, para ello, se usará pd.concat(); y con ingnore_index=True, se vuelve a contar las filas, de 0 a 24. 

capitales = pd.concat([capitales, callao], ignore_index=True)
print(capitales.shape)

(25, 2)


Una vez agregada la fila de Callao, se debe corregir la información de la capital de Lima: En las bases de datos del INEI, PCM, y demás, el distrito de Lima es considerada como la capital del departamento de Lima; por lo cual, para que funciones API, se debe corregir dicha información.

In [8]:
# Con el comando .loc, se podrá cambiar información de la tabla capitales. 

capitales.loc[capitales["departamento"] == "Lima", "capital"] = "Lima"
capitales[capitales["departamento"] == "Lima"]


,departamento,capital
13,Lima,Lima


In [9]:
# Por último, se volverá a usar len() para contar las filas y str.contains() para contar las capitales que tienen paréntesis.

print("Filas:", len(capitales))
print("Capitales con paréntesis:", capitales["capital"].str.contains("(", regex=False).sum())
capitales

Filas: 25
Capitales con paréntesis: 0


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## 2.3. Paso 3

**Enunciado**: Para cada capital, obtengan su latitud y longitud con la API de geocodificación:

https://geocoding-api.open-meteo.com/v1/search?name=Chachapoyas&count=10&language=es

Para ello, se debe aimportar un grupo de paquetes para que se puedan ejecutar los códigos.

In [10]:
import pandas as pd
import requests
from io import StringIO
import time

In [11]:
# Antes de ejecutar el código API, se va probar un primer pedido, solicitando la información de Lima.

url_geo = "https://geocoding-api.open-meteo.com/v1/search"
parametros = {"name": "Lima", "count": 10, "language": "es"}

respuesta_geo = requests.get(url_geo, params=parametros, timeout=30)
print(respuesta_geo.status_code)
print(respuesta_geo.url)

# Para visualizar la información de la respuesta_geo, se usará .json() para convertir la información en un diccionario de Python.

datos = respuesta_geo.json()
pd.DataFrame(datos["results"])[["name", "country_code", "admin1", "latitude", "longitude"]]


200
https://geocoding-api.open-meteo.com/v1/search?name=Lima&count=10&language=es


,name,country_code,admin1,latitude,longitude
0,Lima,PE,Provincia de Lima,-12.04318,-77.02824
1,Lima,US,Ohio,40.74255,-84.10523
2,Lima,US,Nueva York,42.90479,-77.61139
3,Howe,US,Indiana,41.72144,-85.42054
4,Lima,US,Montana,44.63686,-112.59195
5,Lima,US,Illinois,40.17755,-91.37848
6,Stillwater,US,Ohio,40.32340,-81.30844
7,Lima,PY,San Pedro,-23.88496,-56.47891
8,Lima,US,Pensilvania,39.91733,-75.44047
9,Lima,SE,Provincia de Dalarna,60.93333,13.35000


Al ejecutarse la búsquea, se identicia que ha seleccionado la ciudad de Lima de Perú, y de otros países, como Estados Unidos o Paraguay. 

Ya habiendo hecho la prueba pasa Lima, se va consultar por todas las capitales con la función buscar_ciudad

In [12]:
# Primero, se debe definir la función buscar_ciudad(), la cual pedirá a la API todos los lugares que se llaman como la ciudad.

def buscar_ciudad(ciudad):
    """Pide a la API todos los lugares que se llaman como la ciudad."""
    parametros = {"name": ciudad, "count": 10, "language": "es"}
    r = requests.get(url_geo, params=parametros, timeout=30)
    r.raise_for_status()      # si algo falló, avisa aquí y no más adelante
    time.sleep(1)             # pausa de 1 segundo entre pedidos
    return r.json().get("results", [])   # si no encuentra nada, devuelve una lista vacía

# Ahora, se aplicará la función buscar_ciudad() en la tabla capitales.

resultados = capitales["capital"].apply(buscar_ciudad)

# Por último, buscar cuantos resultados hay por ciudad capital.

capitales["n_resultados"] = resultados.apply(len)
capitales


,departamento,capital,n_resultados
0,Amazonas,Chachapoyas,4
1,Áncash,Huaraz,1
2,Apurímac,Abancay,1
3,Arequipa,Arequipa,9
4,Ayacucho,Ayacucho,10
5,Cajamarca,Cajamarca,10
6,Cusco,Cusco,5
7,Huancavelica,Huancavelica,3
8,Huánuco,Huánuco,5
9,Ica,Ica,10


Como se observa, solo 2 ciudades tienen 1 registro, los demás tienen más de un resultado.

## 2.4 Paso 4

**Enunciado**: Verificación: la API puede devolver varias ciudades con el mismo nombre, dentro y fuera del Perú. Impriman una tabla con departamento, capital, el departamento que devolvió la API (admin1), latitud y longitud. Todas las filas deben estar en el Perú y en el departamento correcto. Expliquen cómo eligieron el resultado correcto.

*Pista: cada resultado de la API trae country_code y admin1. Recorran los resultados y quédense con el primero que tenga country_code == "PE". Luego revisen que admin1 coincida con el departamento. admin1 no siempre viene escrito igual ("Departamento de Cusco", "Ancash" sin tilde, "Provincia Constitucional del Callao"), así que revísenlo fila por fila.*


Primero, se debe identificar si alguna de las 25 ciudades capitales ubicados no se encuentra en el código de pais PE.

In [13]:
def primero_en_peru(lista):
    """De la lista de resultados, devuelve el primero que está en el Perú."""
    for resultado in lista:
        if resultado.get("country_code") == "PE":
            return resultado
    return None   # ninguno está en el Perú


#Se identifican cuantas ciudades capitales tienen resultado con el código PE y cuantos no.

elegidos = resultados.apply(primero_en_peru)
print("Capitales CON resultado en el Perú:", elegidos.notna().sum())
print("Capitales SIN resultado en el Perú:", elegidos.isna().sum())


Capitales CON resultado en el Perú: 25
Capitales SIN resultado en el Perú: 0


Entonces, todos los nombres de las ciudades capitales, al menos una vez están registrados con el código de pais PE. 

Según lo  indicado en la tarea, se tiene que solicitar información de admin1, latitud y longitud.

In [14]:
# Ahora, se debe agregar la información de admin1, latitud y longitud a la tabla capitales.

capitales["admin1"] = elegidos.apply(lambda resultado: resultado["admin1"])
capitales["latitud"] = elegidos.apply(lambda resultado: resultado["latitude"])
capitales["longitud"] = elegidos.apply(lambda resultado: resultado["longitude"])

capitales[["departamento", "capital", "admin1", "latitud", "longitud"]]

,departamento,capital,admin1,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


Revisando manualmente se identifica wue la columna departamento y admin1 coinciden. Sin emvargo, si se va comparar con una función, se debe limpiar primero la columna admin1; ya que hay datos adicionales como "Departamenteo de", "Provincia Constitucional del" o las tildes. 

In [15]:
# Con la función normalizar, pasamos el texto a mayúsculas, quita tildes y espacios sobrantes. Para ello, primero se define la función normalizar.

def normalizar(serie):
    """Mayúsculas, sin tildes y sin espacios sobrantes."""
    return (
        serie.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.upper()
        .str.strip()
    )

# Luego, se elimina los términos de admin1 que la API pones antes del nombre de los departamentos.

admin1_limpio = capitales["admin1"].str.replace(
    r"^(Departamento de |Provincia Constitucional del |Provincia de )", "", regex=True
)

# Ya habiendp limpiado, se puede comparar si el admin1 y el departamento son iguales.

capitales["coincide"] = normalizar(admin1_limpio) == normalizar(capitales["departamento"])
capitales[["departamento", "capital", "admin1", "latitud", "longitud", "coincide"]]


,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


El resultado es que todas las comparaciones coinciden (true). Por último, se va mostrar la cantidad de filas que no coinciden entre Departamento y admin1.

In [16]:
print("Filas que coinciden:", capitales["coincide"].sum(), "de", len(capitales))
capitales[~capitales["coincide"]]   # muestra las que NO coinciden


Filas que coinciden: 25 de 25


,departamento,capital,n_resultados,admin1,latitud,longitud,coincide


Para cada capital se recorrieron los resultados de la API y se eligió el primero que tiene country_code == "PE". Después se comparó admin1 con el departamento, quitando 'Departamento de', 'Provincia de', 'Provincia Constitucional del' y las tildes. Las 25 filas coinciden.

## 2.5. Paso 5

**Enunciado**: Para cada capital, pidan la lluvia diaria de su temporada con la API histórica:

https://archive-api.open-meteo.com/v1/archive?latitude=-6.23&longitude=-77.87&start_date=2023-01-01&end_date=2023-05-31&daily=precipitation_sum&timezone=America/Lima

Hagan una pausa de al menos 1 segundo entre pedidos.

Entonces, primero se debe determinar el periodo en el que se va a buscar la información. En el caso de nuestro grupo, es del 1 de enero al 31 de mayo.

In [17]:
# Primero, se debe definir la URL de la API de lluvias, dentro del periodo determinado, del 1 de enero al 31 de mayo de 2025.

url_archivo = "https://archive-api.open-meteo.com/v1/archive"

def pedir_lluvia(latitud, longitud):
    """Pide a la API la lluvia diaria de la temporada 2025 para un punto."""
    parametros = {
        "latitude": latitud,
        "longitude": longitud,
        "start_date": "2025-01-01",
        "end_date": "2025-05-31",
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
   
    # Se hace el pedido a la API y se espera 1 segundo entre pedidos, como pide el enunciado.
    
    r = requests.get(url_archivo, params=parametros, timeout=60)
    r.raise_for_status()      # si algo falló, avisa aquí
    time.sleep(1)             # pausa de 1 segundo entre pedidos (lo pide el enunciado)
    
    # Crear una data frame con información de fechas y lluvias.
    
    tabla = pd.DataFrame(r.json()["daily"])   # la API devuelve fechas y lluvias en "daily"
    tabla.columns = ["fecha", "lluvia_mm"]
    return tabla


In [18]:
# Se prueba la función pedir_lluvia() con las coordenadas de Lima.

prueba = pedir_lluvia(-12.04318, -77.02824)
print(prueba.shape)
prueba.head()


(151, 2)


,fecha,lluvia_mm
0,2025-01-01,0.8
1,2025-01-02,3.0
2,2025-01-03,3.0
3,2025-01-04,0.0
4,2025-01-05,0.0


Del resultado, se puede observar que la tabla tiene 151 filas y 2 columnas, con información de fecha y la lluvia medida en milímetros. En el caso de Lima, se puede ver que la lluvia fue ligera en los primeros días de enero.

In [19]:
lista_lluvias = []

# Se recorre la tabla capitales, y se pide la información de lluvias para cada capital, y se agrega a la lista_lluvias.

for i in capitales.index:
    departamento = capitales.loc[i, "departamento"]
    lluvia = pedir_lluvia(capitales.loc[i, "latitud"], capitales.loc[i, "longitud"])
    lluvia["departamento"] = departamento     # para saber de quién es cada fila
    lista_lluvias.append(lluvia)
    print(departamento, "->", len(lluvia), "días")

# Se juntan las 25 tablas en una sola, una debajo de otra (como con el Callao)
lluvia_diaria = pd.concat(lista_lluvias, ignore_index=True)
print(lluvia_diaria.shape)
lluvia_diaria.head()


Amazonas -> 151 días
Áncash -> 151 días
Apurímac -> 151 días
Arequipa -> 151 días
Ayacucho -> 151 días
Cajamarca -> 151 días
Cusco -> 151 días
Huancavelica -> 151 días
Huánuco -> 151 días
Ica -> 151 días
Junín -> 151 días
La Libertad -> 151 días
Lambayeque -> 151 días
Lima -> 151 días
Loreto -> 151 días
Madre de Dios -> 151 días
Moquegua -> 151 días
Pasco -> 151 días
Piura -> 151 días
Puno -> 151 días
San Martín -> 151 días
Tacna -> 151 días
Tumbes -> 151 días
Ucayali -> 151 días
Callao -> 151 días
(3775, 3)


,fecha,lluvia_mm,departamento
0,2025-01-01,4.6,Amazonas
1,2025-01-02,1.5,Amazonas
2,2025-01-03,5.2,Amazonas
3,2025-01-04,5.3,Amazonas
4,2025-01-05,2.3,Amazonas


Con esto, se confirma que se logró extraer información de todos los departamentos sobre la lluvia, en el periodo indicado, de enero a mayo del año 2025. Es decir, se tiene información de 151 días de registro de cada departamento. La tabla final tiene 3775 filas y 3 columnas. Se realizó la solicitud esperando 1 segundo entre pedidos.

## 2.6. Paso 6

Para cada departamento, calculen:

- lluvia_total_mm: la suma de la lluvia diaria de la temporada.
- dias_lluvia_fuerte: el número de días con 20 mm o más.

Para ello, se usará la función groupby.

In [20]:
# Primero, se prueba la suma de lluvia por departamento

lluvia_diaria.groupby("departamento")["lluvia_mm"].sum()

departamento
Amazonas          427.9
Apurímac          820.0
Arequipa          439.4
Ayacucho          500.8
Cajamarca         875.7
Callao             54.0
Cusco             801.3
Huancavelica      930.4
Huánuco           617.0
Ica               124.3
Junín            1349.1
La Libertad       143.8
Lambayeque        128.1
Lima               54.0
Loreto           2019.9
Madre de Dios    1751.3
Moquegua           97.5
Pasco             874.2
Piura             241.6
Puno             1074.4
San Martín        943.3
Tacna              17.4
Tumbes            699.1
Ucayali          1309.0
Áncash           1415.9
Name: lluvia_mm, dtype: float64

In [21]:
# Luego, se determina cuando una lluvia supera o no los 20 mm

lluvia_diaria["lluvia_fuerte"] = lluvia_diaria["lluvia_mm"] >= 20

# Ahora, se van a calcular los 2 indicadores que solicita la tarea

resumen = lluvia_diaria.groupby("departamento").agg(
    lluvia_total_mm=("lluvia_mm", "sum"),
    dias_lluvia_fuerte=("lluvia_fuerte", "sum"),
).reset_index()

resumen.sort_values("lluvia_total_mm", ascending=False)

,departamento,lluvia_total_mm,dias_lluvia_fuerte
14,Loreto,2019.9,28
15,Madre de Dios,1751.3,30
24,Áncash,1415.9,11
10,Junín,1349.1,15
23,Ucayali,1309.0,21
19,Puno,1074.4,6
20,San Martín,943.3,12
7,Huancavelica,930.4,4
4,Cajamarca,875.7,2
17,Pasco,874.2,1


Se agregaron las columnas lluvia_total_mm y dias_lluvia_fuerte, de acuerdo con lo indicado. Se identifica que el departamento con mayores mm de lluvia fue Loreto, con 2019.9. Además, Madre de Dios es el departamento que más días tuvo con lluvia de 20 mm a más. Tacna fue el departaento que menos llovió, seguido de Lima y Callao, quienen tienen el mismo resultado de 54.0 mm de lluvia; lo cual se debe a que ambos se encuentran muy cerca, por lo cual se encuentran en el mismo punto de la grilla.

## 2.7. Paso 7

**Enunciado**: Verificación: cuenten cuántos días llegaron sin dato (None) y expliquen qué hicieron con ellos. Si el resultado es 0, también es una respuesta válida: lo importante es haberlo comprobado.

Para verificar, se va usar la función .isna().

In [22]:
# Total de días sin dato en toda la tabla
lluvia_diaria["lluvia_mm"].isna().sum()

# Se muestra el resultado con una etiqueta
print("Días sin dato:", lluvia_diaria["lluvia_mm"].isna().sum())

Días sin dato: 0


Entonces, como se observa, la API proporcionó el dato de todos los días, del 1 de enero al 31 de mayo del año 2025. Por ello, no es necesario eliminar ni completar ningún dato.

## 2.8. Paso 8

**Enunciado**: Guarden datos/lluvias_por_departamento.csv con las columnas departamento, capital, latitud, longitud, lluvia_total_mm y dias_lluvia_fuerte.

Debido a que las columnas se encuentran en 2 tablas distintas y comparten un dato en común (departamento), se procederá a unirlos con la función pd.merge.


In [23]:
# De capitales, solo se toman las columnas que pide el enunciado.
tabla_final = pd.merge(
    capitales[["departamento", "capital", "latitud", "longitud"]],
    resumen,
    on="departamento",
)
print(tabla_final.shape)
tabla_final

(25, 6)


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0
1,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11
2,Apurímac,Abancay,-13.63426,-72.88380,820.0,3
3,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3
4,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1
5,Cajamarca,Cajamarca,-7.16378,-78.50027,875.7,2
6,Cusco,Cusco,-13.53188,-71.96701,801.3,4
7,Huancavelica,Huancavelica,-12.78691,-74.97298,930.4,4
8,Huánuco,Huánuco,-9.92882,-76.23989,617.0,4
9,Ica,Ica,-14.07538,-75.73422,124.3,2


Como resultado, se tiene una tabla con 25 filas y 6 columnas. Ahora, se debe guardar en un documento .csv.

In [24]:
tabla_final.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")